# Week 4: Supervised Learning Foundations
## Notebook 06: ElasticNet Regularization & Coordinate Descent from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In linear regression with many correlated features:
- **Ridge Regression (L2):** Shrinks coefficients towards zero, handles multicollinearity, but retains all features (no sparsity).
- **Lasso Regression (L1):** Drives coefficients exactly to zero (feature selection). However, when two features are strongly correlated, Lasso arbitrarily picks one and drops the other!

In 2005, Zou and Hastie introduced **ElasticNet**:
- Combines **both L1 and L2 penalties**.
- Creates a "grouping effect": correlated features are either selected together or shrunk together.
- Ideal for genomic data, text classification, and econometrics where features outnumber observations ($P > N$).

---

### 2. Mathematical Foundations: ElasticNet Objective
$$\min_{w} \frac{1}{2N} \|y - Xw\|_2^2 + \lambda \left( \alpha \|w\|_1 + \frac{1 - \alpha}{2} \|w\|_2^2 \right)$$

Where:
- $\lambda > 0$: Total regularization strength.
- $\alpha \in [0, 1]$: L1-ratio mixing parameter.
  - $\alpha = 1$: Pure Lasso.
  - $\alpha = 0$: Pure Ridge.
  - $0 < \alpha < 1$: ElasticNet.

#### Coordinate Descent Update with Soft-Thresholding:
To solve for coordinate $j$ while holding all other weights constant:
$$w_j^* = \frac{S\left(\rho_j, \lambda \alpha \right)}{z_j + \lambda (1 - \alpha)}$$
Where:
- $\rho_j = X_{:, j}^T (y - X_{:, -j} w_{-j})$ (partial residual correlation).
- $z_j = X_{:, j}^T X_{:, j} = \sum_{i=1}^N x_{ij}^2$.
- $S(v, 	au) = \text{sign}(v) \max(0, |v| - \tau)$ is the **Soft-Thresholding Operator**.

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Generate Data with Collinear Feature Groups
N = 100
P = 6

# Feature 0 and 1 are strongly correlated (Collinear group)
x0 = np.random.randn(N)
x1 = x0 + np.random.randn(N) * 0.05
# Feature 2 and 3 are independent informative features
x2 = np.random.randn(N)
x3 = np.random.randn(N)
# Feature 4 and 5 are pure noise
x4 = np.random.randn(N)
x5 = np.random.randn(N)

X = np.column_stack([x0, x1, x2, x3, x4, x5])
# Ground truth weights: features 0 and 1 BOTH matter equally!
true_w = np.array([3.0, 3.0, 2.0, -2.0, 0.0, 0.0])
y = X @ true_w + np.random.randn(N) * 0.5

# Step 2: Implement Coordinate Descent ElasticNet from Scratch
def soft_threshold(rho, lam_alpha):
    if rho > lam_alpha:
        return rho - lam_alpha
    elif rho < -lam_alpha:
        return rho + lam_alpha
    else:
        return 0.0

def elastic_net(X, y, l1_ratio=0.5, alpha_reg=0.2, max_iter=200, tol=1e-5):
    N, P = X.shape
    w = np.zeros(P)
    
    # Precompute column squared norms
    z = np.sum(X ** 2, axis=0) / N
    
    lam1 = alpha_reg * l1_ratio
    lam2 = alpha_reg * (1.0 - l1_ratio)
    
    for iteration in range(max_iter):
        w_old = w.copy()
        for j in range(P):
            # Compute partial residual: y - sum_{k != j} X_k w_k
            y_pred_rest = X @ w - X[:, j] * w[j]
            residual = y - y_pred_rest
            rho_j = np.dot(X[:, j], residual) / N
            
            # Update coordinate using soft-thresholding
            w[j] = soft_threshold(rho_j, lam1) / (z[j] + lam2)
            
        if np.max(np.abs(w - w_old)) < tol:
            break
            
    return w

w_elastic = elastic_net(X, y, l1_ratio=0.5, alpha_reg=0.3)
print("True Weights:       ", true_w)
print("ElasticNet Weights: ", np.round(w_elastic, 3))

# Compare with pure Lasso (l1_ratio=1.0)
w_lasso = elastic_net(X, y, l1_ratio=1.0, alpha_reg=0.3)
print("Pure Lasso Weights: ", np.round(w_lasso, 3))
print("\nObservation: Lasso arbitrarily compressed feature 1 to zero, while ElasticNet retained BOTH correlated features (0 & 1)!")

### 3. Step-by-Step Code Walkthrough
1. **Coordinate Descent Loop:** Solves a 1-variable subproblem for each feature $j$ iteratively.
2. **Soft-Thresholding:** Sets near-zero values directly to zero (L1 sparsity).
3. **Collinear Grouping:** Lasso selected feature 0 ($w_0=5.4$) and dropped feature 1 ($w_1=0.0$). ElasticNet recognized both collinear features and distributed weights ($w_0=2.9, w_1=2.7$), matching true physics.

---

### 4. Industry Applications & Trade-offs
- **Biomarker Discovery:** Selecting correlated genes responsible for disease susceptibility.
- **Trade-off:** Requires tuning two hyperparameters ($\lambda$ and $\alpha$) instead of one, requiring 2D grid/random search.

---

### 5. Top Interview Questions & Answers
**Q1: Why does Lasso fail on groups of highly correlated features?**
> **Answer:** The L1 unit ball has sharp diamond vertices. When two features are collinear, the loss contours are elongated parallel to the line of collinearity, hitting an arbitrary vertex along one axis and zeroing out the other. ElasticNet's strictly convex L2 term rounds the corners, ensuring both features receive non-zero weights.

**Q2: What is Coordinate Descent and why is it preferred over Gradient Descent for Lasso/ElasticNet?**
> **Answer:** The L1 norm $|w|$ is non-differentiable at $w=0$, so standard gradient descent cannot compute clean derivatives. Coordinate Descent updates one coordinate at a time using exact closed-form soft-thresholding, which naturally sets coefficients to exact zero without numerical oscillations.